# Direct LLM interpretation of temporal merging metrics

Save this notebook in `drone-dataset-tools-master/src`. It runs a small synchronous pilot directly from Jupyter. Use the Batch API later for the full dataset.

**Important:** point `input_csv` to a long-format temporal-metric file, not to the event-level combined summary CSV.

In [ ]:
%pip install -U openai pandas


## 1. Imports and API key

The key is entered invisibly and kept only in the current kernel session.

In [ ]:
from pathlib import Path
from getpass import getpass
import json
import os
import time

import pandas as pd
from openai import OpenAI

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")

client = OpenAI()
print("OpenAI client created.")


## 2. Paths and experimental settings

Change `input_csv` to the actual temporal-metric CSV. The notebook assumes it is saved somewhere under `src/results`.

In [ ]:
# Replace this with the actual long-format temporal-metric CSV.
input_csv = Path("results/temporal_metrics_all_vehicles.csv")
output_jsonl = Path("results/llm_interpretations_pilot.jsonl")

MODEL = os.environ["OPENAI_MODEL"]
REASONING_EFFORT = "low"
FRAME_RATE = 25.0
STRIDE = 1            # Keep every row for the main experiment.
PILOT_LIMIT = 10      # Start small.

print("Current working directory:", Path.cwd())
print("Input CSV:", input_csv.resolve())
print("Input exists:", input_csv.exists())
print("Output JSONL:", output_jsonl.resolve())


## 3. Prompt and structured-output schema

In [ ]:
SYSTEM_PROMPT = 'You are a traffic-behavior analyst examining one highway on-ramp merging maneuver from time-ordered numerical metrics.\n\nInterpret only the supplied temporal metric-level inputs. You are not given cluster membership, cluster summaries, event-level aggregate indicators, manually assigned behavior labels, or driver intent.\n\nRules:\n1. Respect temporal order, duration, persistence, changes, and coincidence among metrics.\n2. Treat larger PCAD as greater modeled collision-avoidance difficulty, but do not invent universal thresholds.\n3. Describe longitudinal velocity as maintained, increasing, decreasing, fluctuating, or constrained. Do not judge absolute efficiency without a reference speed or traffic context.\n4. Describe comfort from the timing, magnitude, persistence, and coincidence of longitudinal and lateral jerk. Do not invent universal thresholds.\n5. Explain temporal relationships among safety, speed, and comfort, but do not claim causality from coincidence alone.\n6. Cite approximate times and values from the input as evidence.\n7. State limitations and reduce confidence when the data are incomplete, too short, noisy, or contradictory.\n8. Create a concise free-form behavior label. Do not select from a predefined taxonomy.\n9. Return only the required structured output.'

OUTPUT_SCHEMA = {'type': 'object', 'properties': {'behavior_label': {'type': 'string'}, 'behavior_description': {'type': 'string'}, 'safety_interpretation': {'type': 'string'}, 'efficiency_interpretation': {'type': 'string'}, 'comfort_interpretation': {'type': 'string'}, 'interaction_interpretation': {'type': 'string'}, 'evidence': {'type': 'array', 'items': {'type': 'object', 'properties': {'time_range_s': {'type': 'string'}, 'observed_values': {'type': 'string'}, 'supports': {'type': 'string'}}, 'required': ['time_range_s', 'observed_values', 'supports'], 'additionalProperties': False}}, 'limitations': {'type': 'array', 'items': {'type': 'string'}}, 'confidence': {'type': 'string', 'enum': ['low', 'medium', 'high']}}, 'required': ['behavior_label', 'behavior_description', 'safety_interpretation', 'efficiency_interpretation', 'comfort_interpretation', 'interaction_interpretation', 'evidence', 'limitations', 'confidence'], 'additionalProperties': False}


## 4. Load and standardize the temporal data

Recognized aliases include `recordingId`, `trackId`, `lonVelocity`, `pcad_max`, `lon_jerk`, and `lat_jerk`.

In [ ]:
COLUMN_ALIASES = {
    "recording_id": ["recording_id", "recordingId"],
    "track_id": ["track_id", "trackId", "ego_track_id"],
    "time_s": ["time_s", "time", "timestamp_s"],
    "frame": ["frame"],
    "pcad_max_mps": ["pcad_max_mps", "pcad_max", "frame_level_pcad"],
    "critical_position": ["critical_position", "critical_relation"],
    "lon_velocity_mps": ["lon_velocity_mps", "lon_velocity", "lonVelocity"],
    "lon_jerk_mps3": ["lon_jerk_mps3", "lon_jerk", "longitudinal_jerk"],
    "lat_jerk_mps3": ["lat_jerk_mps3", "lat_jerk", "lateral_jerk"],
}

def find_column(df, aliases):
    return next((name for name in aliases if name in df.columns), None)

def canonicalize_columns(df, frame_rate=25.0):
    rename = {}
    for canonical, aliases in COLUMN_ALIASES.items():
        found = find_column(df, aliases)
        if found is not None:
            rename[found] = canonical

    out = df.rename(columns=rename).copy()

    for column in ["recording_id", "track_id"]:
        if column not in out.columns:
            raise ValueError(f"Missing required identity column: {column}")

    if "time_s" not in out.columns:
        if "frame" not in out.columns:
            raise ValueError("Provide either time_s or frame.")
        out["time_s"] = out.groupby(["recording_id", "track_id"])["frame"].transform(
            lambda s: (pd.to_numeric(s, errors="coerce") - pd.to_numeric(s, errors="coerce").min()) / frame_rate
        )

    required_metrics = ["pcad_max_mps", "lon_velocity_mps", "lon_jerk_mps3", "lat_jerk_mps3"]
    missing = [c for c in required_metrics if c not in out.columns]
    if missing:
        raise ValueError(f"Missing temporal metric columns: {missing}")

    if "critical_position" not in out.columns:
        out["critical_position"] = "Unavailable"

    return out

raw_df = pd.read_csv(input_csv)
temporal_df = canonicalize_columns(raw_df, frame_rate=FRAME_RATE)

print("Rows:", len(temporal_df))
print("Vehicles:", temporal_df[["recording_id", "track_id"]].drop_duplicates().shape[0])
print("Columns:", temporal_df.columns.tolist())
temporal_df.head()


## 5. Build one vehicle's prompt

In [ ]:
INPUT_COLUMNS = [
    "time_s",
    "pcad_max_mps",
    "critical_position",
    "lon_velocity_mps",
    "lon_jerk_mps3",
    "lat_jerk_mps3",
]

def prepare_vehicle_table(vehicle_df, stride=1):
    table = vehicle_df.sort_values("time_s").iloc[::stride][INPUT_COLUMNS].copy()
    for column in [c for c in INPUT_COLUMNS if c != "critical_position"]:
        table[column] = pd.to_numeric(table[column], errors="coerce").round(3)
    table["critical_position"] = table["critical_position"].fillna("Unavailable").astype(str)
    return table

def build_vehicle_prompt(recording_id, track_id, table):
    csv_text = table.to_csv(index=False, lineterminator="\n", float_format="%.3f")
    return (
        "Analyze this highway on-ramp merging maneuver.\n\n"
        f"recording_id: {recording_id}\n"
        f"track_id: {track_id}\n"
        "Rows are ordered by elapsed time.\n\n"
        "Temporal metric table (CSV):\n"
        "```csv\n"
        f"{csv_text}"
        "```"
    )


## 6. Run one test vehicle

Inspect this result carefully before running more vehicles.

In [ ]:
def interpret_vehicle(recording_id, track_id, vehicle_df):
    table = prepare_vehicle_table(vehicle_df, stride=STRIDE)
    user_input = build_vehicle_prompt(recording_id, track_id, table)

    response = client.responses.create(
        model=MODEL,
        instructions=SYSTEM_PROMPT,
        input=user_input,
        reasoning={"effort": REASONING_EFFORT},
        text={
            "verbosity": "medium",
            "format": {
                "type": "json_schema",
                "name": "merging_behavior_interpretation",
                "schema": OUTPUT_SCHEMA,
                "strict": True,
            },
        },
        max_output_tokens=1800,
        store=False,
    )

    return json.loads(response.output_text), response

first_recording_id, first_track_id = (
    temporal_df[["recording_id", "track_id"]]
    .drop_duplicates()
    .iloc[0]
    .tolist()
)

first_vehicle = temporal_df[
    (temporal_df["recording_id"] == first_recording_id)
    & (temporal_df["track_id"] == first_track_id)
]

test_interpretation, test_response = interpret_vehicle(
    first_recording_id,
    first_track_id,
    first_vehicle,
)

print(json.dumps(test_interpretation, indent=2, ensure_ascii=False))
print("Response ID:", test_response.id)
print("Usage:", test_response.usage)


## 7. Run a small pilot and save JSONL

This cell appends results after every vehicle, so completed results remain available if the notebook stops.

In [ ]:
vehicle_keys = (
    temporal_df[["recording_id", "track_id"]]
    .drop_duplicates()
    .sort_values(["recording_id", "track_id"])
    .head(PILOT_LIMIT)
)

output_jsonl.parent.mkdir(parents=True, exist_ok=True)

with output_jsonl.open("w", encoding="utf-8") as output_file:
    for _, key in vehicle_keys.iterrows():
        recording_id = key["recording_id"]
        track_id = key["track_id"]

        vehicle_df = temporal_df[
            (temporal_df["recording_id"] == recording_id)
            & (temporal_df["track_id"] == track_id)
        ]

        record = {
            "recording_id": recording_id,
            "track_id": track_id,
            "model": MODEL,
            "reasoning_effort": REASONING_EFFORT,
        }

        try:
            interpretation, response = interpret_vehicle(recording_id, track_id, vehicle_df)
            record.update({
                "status": "ok",
                "response_id": response.id,
                "interpretation": interpretation,
            })
            print(f"OK: recording {recording_id}, track {track_id}")
        except Exception as exc:
            record.update({"status": "error", "error": repr(exc)})
            print(f"ERROR: recording {recording_id}, track {track_id}: {exc}")

        output_file.write(json.dumps(record, ensure_ascii=False) + "\n")
        output_file.flush()
        time.sleep(0.2)

print("Saved pilot results to:", output_jsonl.resolve())


## 8. Read the saved results

In [ ]:
results_df = pd.read_json(output_jsonl, lines=True)
results_df.head()
